In [0]:
student=spark.read.csv("/Volumes/course_enrollment/online_course_enroll/filestorage/student.csv").toDF("student_id","student_name","blood_group","phone_no","city","student_status")
course=spark.read.csv("/Volumes/course_enrollment/online_course_enroll/filestorage/course.csv").toDF("course_id","course_name")
enrollment=spark.read.csv("/Volumes/course_enrollment/online_course_enroll/filestorage/enrollment.csv").toDF("enrollment_id","student_id","course_id","enrollment_date","progress_percentage","enrollment_status")
display(student)
display(course)
display(enrollment)


student_id,student_name,blood_group,phone_no,city,student_status
101,Alice Smith,O+,9876543210,Chennai,Active
102,Bob Jones,A+,9876543211,Mumbai,Active
103,null,B+,9876543212,Bangalore,Active
104,Charlie Brown,O-,9876543213,Delhi,Active
105,Diana Prince,AB+,9876543214,Hyderabad,Active
106,Evan Wright,A-,9876543215,Kochi,Active
107,Fiona Gallagher,O+,9876543216,Pune,Active
108,George Clark,B-,9876543217,Chennai,Active
109,Hannah Abbott,A+,9876543218,Mumbai,Active
110,Ian Malcolm,AB-,9876543219,Bangalore,Active


course_id,course_name
201,Introduction to Python
202,Data Science Basics
203,Machine Learning


enrollment_id,student_id,course_id,enrollment_date,progress_percentage,enrollment_status
1001,101,201,2026-01-15,85,Enrolled
1002,102,202,2026-01-20,45,Enrolled
1003,103,201,2026-02-01,100,Completed
1004,104,203,2026-01-10,12,Dropped
1005,105,202,null,105,Enrolled
1006,106,201,2026-02-15,null,Enrolled
1007,107,203,2026-01-22,100,Completed
1008,108,202,2026-02-05,-5,Dropped
1009,109,201,2026-01-18,50,Enrolled
1010,110,203,2026-02-10,75,Enrolled


In [0]:
student=student.na.fill("Not Provided",subset=["student_name"])
display(student)

student_id,student_name,blood_group,phone_no,city,student_status
101,Alice Smith,O+,9876543210,Chennai,Active
102,Bob Jones,A+,9876543211,Mumbai,Active
103,Not Provided,B+,9876543212,Bangalore,Active
104,Charlie Brown,O-,9876543213,Delhi,Active
105,Diana Prince,AB+,9876543214,Hyderabad,Active
106,Evan Wright,A-,9876543215,Kochi,Active
107,Fiona Gallagher,O+,9876543216,Pune,Active
108,George Clark,B-,9876543217,Chennai,Active
109,Hannah Abbott,A+,9876543218,Mumbai,Active
110,Ian Malcolm,AB-,9876543219,Bangalore,Active


In [0]:
from pyspark.sql.functions import col,when
enrollment=enrollment.na.fill({"enrollment_date":"Not Provided","progress_percentage":0})
display(enrollment)
enrollment= enrollment.withColumn(
    "progress_percentage",
    when(col("progress_percentage") < 0, 0)
     .when(col("progress_percentage") > 100, 100)
     .otherwise(col("progress_percentage"))
)
display(enrollment)

enrollment_id,student_id,course_id,enrollment_date,progress_percentage,enrollment_status
1001,101,201,2026-01-15,85,Enrolled
1002,102,202,2026-01-20,45,Enrolled
1003,103,201,2026-02-01,100,Completed
1004,104,203,2026-01-10,12,Dropped
1005,105,202,Not Provided,100,Enrolled
1006,106,201,2026-02-15,0,Enrolled
1007,107,203,2026-01-22,100,Completed
1008,108,202,2026-02-05,0,Dropped
1009,109,201,2026-01-18,50,Enrolled
1010,110,203,2026-02-10,75,Enrolled


enrollment_id,student_id,course_id,enrollment_date,progress_percentage,enrollment_status
1001,101,201,2026-01-15,85,Enrolled
1002,102,202,2026-01-20,45,Enrolled
1003,103,201,2026-02-01,100,Completed
1004,104,203,2026-01-10,12,Dropped
1005,105,202,Not Provided,100,Enrolled
1006,106,201,2026-02-15,0,Enrolled
1007,107,203,2026-01-22,100,Completed
1008,108,202,2026-02-05,0,Dropped
1009,109,201,2026-01-18,50,Enrolled
1010,110,203,2026-02-10,75,Enrolled


In [0]:
joined_df=student.join(enrollment,student.student_id==enrollment.student_id,"inner").join(course,enrollment.course_id==course.course_id,"inner")
final_df=joined_df.select("student_name","course_name","enrollment_date","progress_percentage","enrollment_status")
display(final_df)

student_name,course_name,enrollment_date,progress_percentage,enrollment_status
Alice Smith,Introduction to Python,2026-01-15,85,Enrolled
Bob Jones,Data Science Basics,2026-01-20,45,Enrolled
Not Provided,Introduction to Python,2026-02-01,100,Completed
Charlie Brown,Machine Learning,2026-01-10,12,Dropped
Diana Prince,Data Science Basics,Not Provided,100,Enrolled
Evan Wright,Introduction to Python,2026-02-15,0,Enrolled
Fiona Gallagher,Machine Learning,2026-01-22,100,Completed
George Clark,Data Science Basics,2026-02-05,0,Dropped
Hannah Abbott,Introduction to Python,2026-01-18,50,Enrolled
Ian Malcolm,Machine Learning,2026-02-10,75,Enrolled


In [0]:
final_df.write.format("delta").mode("overwrite").save("/Volumes/course_enrollment/online_course_enroll/filestorage/final_df")